<center>
    <p>115 Theoretical Neuroscience II</p>
    <h1></h1>
    <h1>Lecture 01:</h1>
    <h1>Firing rate models</h1>
    <p>----</p>
    <p>Prof. Jochen Braun Ph.D.</p>
    <p>Institute of Biology</p>
    <p>Otto-von-Guericke University Magdeburg</p>
    <p>----</p>
    <p>Textbook:</p>
    <p>Peter Dayan & Larry Abbott (2001) Theoretical Neuroscience, MIT Press.</p>
</center>

In [ ]:
from IPython.display import display, Math

# Inject custom MathJax configuration for additional packages
custom_latex = r"""
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage{eufrak}
"""

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/Schedule SS2026.png" width="1200">

# **Abstract:**

We seek to analyze and model the activity of many neurons in a population.
To this end, we simplify and consider only the average **firing rate** of neurons in a population. This makes sense when the population is 'homogeneous' in that external input modulates all neurons similarly.

Given the neuron models of the last semester, how should we expect population activity to change with external input?  We analyze this in terms of **average synaptic current** and **activation function**, and find **two successive low-pass filters**: synaptic time-constant and effective membrane time-constant.

Lastly, we begin to consider **multiple populations**, linked by synaptic projections.  A key role is played by connectivity (synaptic weights), which is conveniently formalized in terms of **Linear Algebra**.  Feedforward and recurrent networks behave very differently and require different methods of analysis (see next lectures).


# **Overview:**

1. **Multi-unit and single-unit activity**

2. **Firing rate or 'population activity'**

2. **Firing rate model for one population**

3. **Firing rate models for interacting populations**


# **Credits:**

Mathew Diamond, SISSA Trieste

# **1. Multi-unit and single-unit activity**

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_01.png" width="800">

Mathew Diamond, Trieste

(Slide-1)

# Record multi-unit activity (MUA) during whisker task

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_02.png" width="800">

(Slide-2)

# Somatocortical domains of individual whiskers

Rat brain, barrel cortex (yellow patches), and electrode array (red square).

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_03.png" width="800">

Whiskers on snout of rat.

(Slide-3)



# Implanted electrode array

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_04.png" width="800">

(Slide-4)

# Post-mortem reconstruction

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_05.png" width="800">

(Slide-5)

# MUA from 6 electrodes

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_06.png" width="800">

(Slide-6)

Extract single-unit activity

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_07.png" width="800">

(Slide-7)


# Spike-sorting identifies single-units

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_08.png" width="800">

(Slide-8)

# Whisker-evoked multi-unit activity

Comparing different whisker domains, we find that multi-unit firing is modulated selectively. A particular whisker stimulus will evoke firing from units recorded from some electroders, but not from other electrodes.

As repeated stimulation of the same whiskers evokes similar firing responses (similar electrodes and spike rates), it makes sense to average the observed firing over repetitions ('trials').

The histograms below show for each of 100 electrodes the average spike count in 8 time windows of 5 ms duration (total duration 40 ms). This type of representation is called a **post-stimulus time histogram (PSTH)**.



<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Diamond_09.png" width="800">

(Slide-9)

# Summary multi-unit and single-unit

- Typical electrodes are extracellular and may record the firing of multiple units in the vicinity.

- If desired, the firing of different units can be distinguished in terms of spike shape ('spike sorting').

- Often, most of the units recorded from one electrode respond similarly to sensory stimulation, in which case their responses may be combined.

- A classical example are the whisker domains of barrel cortex, a prominent part of the primary somatosensory cortex in rodents.

(Slide-10)







# **2. Firing rate or population activity**

When sensory stimuli are repeated exactly, the spiking responses of neurons are similar, but not identical.  Specifically, sensory stimulation seems to control the probability of spiking, rather than the emission of individual spikes.

For this reason, the spiking responses of neurons are often reported in two formats:

- **Raster plot:** Individual spikes are shown as rows of small dots or lines along a time axis.  Repeated trials are shown as mulitple rows, one above the other, all aligned to the stimulus onset.

- **Post-stimulus time histogram, PSTH:** The stimulus duration is divided into short time windows (5 ms or 10 ms 'bins') and the number of spikes per bin is averaged over all trials. The resulting histogram measures *firing rate, firing probability, firing density* in units of spikes per second.


<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Rasterplot.png" width="800">

Spiking response of a single neuron to repeated presentation of exactly identical visual stimulation with dynamic random dots (see WS Lecture 13). Top: 100% coherent, middle: 50% coherent, bottom: 0% coherent.

D&A Fig-1-19, adapted from [Bair & Koch (1996)](https://core.ac.uk/reader/216124520?utm_source=linkout)

(Slide-11)



In [ ]:
from inspect import signature
# @title Spike raster and firing rate {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

# histogram equalization for flat distribution
def RandomRateModulation( Tend, dt, sigma, Npeak, Rpeak ):

    # time axis in seconds
    t_i   = np.linspace( 0, Tend, int(Tend/dt), endpoint=True );
    t_min = t_i[0];
    t_max = t_i[-1];

    tpeak_k = t_min + (t_max-t_min) * np.random.rand(Npeak);

    # firing rate in spikes per second
    r_i = np.full( len(t_i), 10 );

    for k in range( len(tpeak_k) ):

        t_peak = tpeak_k[k];

        r_i = r_i + Rpeak * np.exp(-(t_i-t_peak)**2/(2*sigma**2) );

    return t_i, r_i

def MakeSpikeRaster( t_i, r_i, Nrepeat ):

    # time axis interval
    dt    = ( t_i[1]-t_i[0] );

    # spike probability per interval
    p_i   = r_i * dt;

    # repeated trials
    p_ki = np.tile( p_i, (Nrepeat, 1))

    ran_ki = np.random.rand( Nrepeat, len(t_i) )

    # spike flag per simulation interval and trial (0 or 1)
    s_ki =  ran_ki < p_ki;

    return s_ki

def GetFiringRate( t_i, s_ki, Nbin ):

    # trial number
    Ntrial = s_ki.shape[0];

    # bin times
    tbin_j = np.linspace( 0, t_i[-1], Nbin );

    # bin width
    dtbin = tbin_j[1] - tbin_j[0];

    # spike counts
    n_kj = np.zeros( (Ntrial, Nbin) );
    for j in range( Nbin-1 ):
        ix = np.where( (t_i >= tbin_j[j]) & (t_i < tbin_j[j]+dtbin) );

        s_kix = np.squeeze( s_ki[:,ix] )

        n_k = np.sum( s_kix, axis=1 );

        n_kj[:,j] = n_k;

    # firing rate
    f_kj = n_kj / dtbin;

    # average over repetitions
    f_j  = np.mean( f_kj, axis=0 );

    return tbin_j, dtbin, f_j


def plot_spike_raster_and_firing_rate( Tend=1.0, dtms=1, Nbin = 50, Rpeak=40.0, Nrepeat=100 ):
    """(Slide-12)"""

    # number and width of rate peaks
    Npeak = 2;
    sigma = 0.1;

    # convert dt to seconds
    dt = dtms / 1000;

    # timecourse of rate modulation
    (t_i, r_i) = RandomRateModulation( Tend, dt, sigma, Npeak, Rpeak )

    # spike raster
    s_ki = MakeSpikeRaster( t_i, r_i, Nrepeat )

    # get average firing rate per bin
    (tbin_j, dtbin, f_j) = GetFiringRate( t_i, s_ki, Nbin )

    # Plot results
    fig, axs = plt.subplots( 2,1, figsize=(12,8))

    # first subplot

    for k in range( Nrepeat):
        ix = np.where( s_ki[k,:] == 1);
        xix = t_i[ix]
        yix = np.ones( len(ix[0]) ) * k;

        axs[0].plot( xix, yix, 'k.', markersize=5 )

    axs[0].set_xlim((0,Tend))
    axs[0].set_xlabel('time [s]')
    axs[0].set_box_aspect(1/4)
    axs[0].set_ylabel('Trials')
    axs[0].set_title('Spike raster', fontsize=24);

    # second subplot

    axs[1].bar(tbin_j, f_j, width=0.8*dtbin, color='black')

    axs[1].set_xlim((0,Tend))
    axs[1].set_xlabel('time [s]')
    axs[1].set_box_aspect(1/4)
    axs[1].set_ylabel('Spikes per second')
    axs[1].set_title('Firing rate', fontsize=24);


    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge
interact(plot_spike_raster_and_firing_rate, Tend = (1.0, 3.0, 0.01 ), dtms = (0.5, 5, 0.1), Nbin=(50, 200, 50), Rpeak=(10,100,10), Nrepeat=(1,100,1 ) )


interactive(children=(FloatSlider(value=1.0, description='Tend', max=3.0, min=1.0, step=0.01), FloatSlider(val…

<function __main__.plot_spike_raster_and_firing_rate(Tend=1.0, dtms=1, Nbin=50, Rpeak=40.0, Nrepeat=100)>

# Firing rate defined as 'spike density'

Spike density is average number of spikes per unit time.  For example, recording a neuron spiking at $4 Hz$ over a $2 s$ trial, you expect to count $8$ spikes.  In $50$ trials, you expect $400$ spikes.  Now average 'sliding windows' of different duration over $50$ trials:

<br>

$$\begin{eqnarray}
\begin{array}{ccccc}
\textit{size} & \textit{ave. count} & \textit {comb. size} & \textit {comb. count} & \textit{spike density}\\
5~ms & 0.02 & 250~ms & 1 & 4~Hz\\
20~ms & 0.08 & 1~s & 4 & 4~Hz\\
50~ms & 0.2 & 2.5~s & 10 & 4~Hz\\  
\end{array}
\end{eqnarray}$$

<br>

The post-stimulus time-histogram is computed from the number of spikes per bin, averaged over trials, and divided by bin width. The units are spikes per second or Hz.

(Slide-13)

# Population activity

A typical network of cortical neurons involves at a minimum some millions of excitatory and inhibitory neurons (150,000 per $mm^2$), each with complex dendrites and axons, with many types of membrane channels for Na$^+$, K$^+$, Ca$^{2+}$, etc. and receiving input and emitting output through ~8,000 synapses on average.   Models of this scale are extremely expensive and unwieldy (many time-scales!).  

<br>

To simplify, we can describe activity of *homogeneous populations* of neurons in terms of the *average firing rate*.   Here 'homogeneous' means that external stimuluation modulates the firing rates of individual neurons similarly (i.e., proportionally to the modulatio of the population rate).

<br>

In this approximation, **population activity** is defined as the average firing rate of neurons.

(Slide-14)


# Asynchronous and irregular firing

Within cortical columns, such as whisker domains, the neurons are typically modulated similarly by sensory input, so that population activity is a meaningful description.

<br>

Neurons in cortical columns are densely interconnected and typically fire in an irregular and asynchronous mode. In other words, sensory input modulates firing probabilities rather than individual spike times.

<br>

'Asynchronous irregular' firing is a consequence of the sparse and random connectivity between cortical neurons.

<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Columns.png" width="600">

(Slide-15)

# Ergodicity of homogeneous populations

An average over trials (one neuron, many trials, easy to record!), is often assumed to approximate an average over neurons (many neurons, one trial, far more difficult to record!).

<br>

This 'ergodicity'  provides an alternative definition for a 'homogeneous population'.

<br>

In sensory cortices with columnar organisation, 'ergodicity' is often a reasonable assumption.

(Slide-16)

# Points to note

- Typically, sensory stimulation modulates the firing rate (firing probability) of cortical neurons, rather than controlling the timing of individual spikes.


- In columns of sensory cortex, neurons are typically homogenous in that they exhibit similar response properties and in that sensory stimulation modulates all firing rates proportionally.


- In homogeneous populations, we can assume *ergodicity*: trial average equals population average.

- The firing activity of homogeneous populations is described by the instantaneous **firing rate** (spike density = average spike count per unit time) and its modulation over time (post-stimulus time histogram, PSTH).

(Slide-17)

# **3. Firing rate model for one population**

Consider a single neuron with pre-synaptic spikes (input), total synaptic currents, and post-synaptic spikes (output). Modeling individual neurons offers a *microscopic level of description*.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Rate_to_rate_a.png" width="800">

<br>

Now consider a homogeneous population of neurons, each with its own inputs, synaptic currents, and outputs.

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Rate_to_rate_c.png" width="800">

(The illustration is misleading: different neurons receive and emit spikes asynchronously!)

We seek a *mesoscopic description* of this population that could model how the collective inputs are transformed into the collective outputs.

(Slide-18)


# Population averages

Let's define, for each neuron $i$, the time-varying *input activity* (firing rate) $u_i(t)$, the time-varying synaptic current $I^\mathit{syn}_i(t)$, and the time-varying *output activity* (firing rate) $v_i(t)$.

Averaging over all $N$ neurons in our homogeneous population, gives us population averages $u(t)$, $I_s(t)$, and $v(t)$

$$\begin{eqnarray}
u(t) = \frac{1}{N} \sum_i u_i(t), \qquad I_s(t) = \frac{1}{N} \sum_i I^\mathit{syn}_i(t), \qquad v(t) = \frac{1}{N} \sum_i v_i(t)
\end{eqnarray}$$

This gives us mean input activity (firing rate) $u(t)$, mean synaptic current $I_s(t)$, and mean output activity (firing rate) $v(t)$.

<br>

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_Rate_to_rate_d.png" width="400">

<br>

A firing rate model is formulated in terms of such average population activities (mesoscopic description). The goal of a firing rate model is to understand how input activity translates into output activity.

This problem can be divided into two parts:


- How does $I_s(t)$ depend on $u(t)$ ?


- How does $v(t)$ depend on $I_s(t)$ ?

(Slide-19)

# Input rate $u$ and synaptic current $I_s$

We assume independent (Poisson distributed) input spikes and individual neuron-and-synapse currents $i_s(t)$ with an exponential decay

$$\begin{eqnarray}
i_s(t) = \frac{w_s}{\tau_s} \, \exp\left(-\frac{t}{\tau_s} \right), \qquad t \geq 0
\end{eqnarray}$$

where $t$ is time after the synaptic event, $w_s$ is synaptic charge, and $\tau_s$ the synaptic time constant.

As the rate of input spikes equals the rate of synapses being triggered, the input rate and the synaptic current are related by a convolution:

$$\begin{eqnarray}
I_s(t) =   \int_{-\infty}^t i_s(t-t') \, u(t') \, dt'  \quad \Leftrightarrow \quad
\end{eqnarray}$$

$$\begin{eqnarray}
\frac{dI_s}{dt} &=& i_s(0) \, u(t) + \int_{-\infty}^t \left[\frac{\partial}{\partial t}i_s(t-t')\right] \, u(t') \, dt' =
\\
&=& \frac{w_s }{\tau_s}\, u(t) -\frac{1}{\tau_s} \, \int_{-\infty}^t i_s(t-t') \, u(t') \, dt' =
\\
&=& \frac{1}{\tau_s} \, \left( w_s \, u - I_s \right)
\end{eqnarray}$$

Thus, we conclude that $I_s(t)$ is a low-pass-filtered version of input activity $u(t)$

$$\begin{eqnarray}
\tau_s \,  \frac{dI_s(t)}{dt} = - I_s(t) + w_s \, u(t)
\end{eqnarray}$$

This dynamic equation is our familiar 'exponential relaxation' from the WS. As long as $w_s \, u$ is larger (smaller) than $I_s$, $I_s$ increases (decreases) until it equals $w_s \, u$.  In other words, this dynamic predicts the existence of a **steady-state**:

$$\begin{eqnarray}
0\stackrel{!}{=} \tau_s \,  \frac{dI_s(t)}{dt} = - I_s(t) + w_s \, u(t)\qquad\Rightarrow\qquad I_s = w_s u
\end{eqnarray}$$

$I_s$ stops changing at the steady-state value $w_s u$.

(Slide-20)

# Synaptic weight

The synaptic weight $w_s$ that scales $u(t)$ is the average electric charge delivered per presynaptic spike (current $\times$ time)

$$\begin{eqnarray}
\tau_s \, \frac{dI_s}{dt} = -I_s +  w_s \, u(t)
\end{eqnarray}$$

and $\tau_s$ is the synaptic time-constant.

<br>

We have now modeled how input activity $u(t)$ affects the instantaneous synaptic current $I_s(t)$ of a homogeneous population.

The key idea was to rely on the shape of synaptic potentials, which is approximately exponential.

(Slide-21)

# Activation function $F(I_s)$

For a constant synaptic current $I_s$, the output activity (firing rate) can be assumed to approach some steady-state value $v_{ss}$, which typically changes with $I_s$.

*Mesoscopically*, the relation between $I_s$ and $v_{ss}$ defines a function $F()$

$$\begin{eqnarray}
v_{ss} = F(I_s)
\end{eqnarray}$$

which is called the **activation function**. *Microscopically*, the activation function can be thought of as describing the increasing probability of output firing, as synaptic currents move fluctuating membrane potentials closer to threshold (see Lecture 7 of WS).

Typically, the activation function $F()$ is monotonically increasing and sigmoidal (with threshold, inflection point, and asymptotic saturation).

The activation function $F()$ translates a constant synaptic current into a steady-state output firing rate.

(Slide-22)

# Output rate $v(t)$ and synaptic current $I_s(t)$

For time-dependent synaptic currents $I_s(t)$, the neuronal membrane capacitance and resistance act as (yet another) low-pass filter.  In other words, the *membrane potential* is a low-pass-filtered version of the synaptic current.  Accordingly, we expect the time-dependent output rate to be a low-pass-filtered version of the (instantaneous) steady-state value $v_{ss}(t)$:

<br>

$$\begin{eqnarray}
\tau_r \, \frac{dv(t)}{dt} = - v(t) + v_{ss}(t) = - v(t) + F\left[ I_s(t) \right]
\end{eqnarray}$$

<br>

The effective time constant $\tau_r$ does not relate directly to membrane properties.  Typically, its value is much smaller than the membrane time-constant.

<br>

This dynamic equation is another 'exponential relaxation' with the **steady-state**

$$\begin{eqnarray}
0\stackrel{!}{=} \tau_r \,  \frac{dv(t)}{dt} = - v(t) + F\left[ I_s(t) \right]\qquad\Rightarrow\qquad v_{ss} = F\left[ I_s(t) \right]
\end{eqnarray}$$

(Slide-23)


# Summary so far

We have now related input activity $v(t)$, synaptic current $I_s(t)$ and output activity $u(t)$ in terms of two dynamic equations (both low-pass filters, or exponential relaxations):

$$\begin{eqnarray}
\tau_s \, \frac{dI_s}{dt} = -I_s +  w_s \, u(t)
\end{eqnarray}$$

with synaptic time-constant $\tau_s$ and synaptic weight $w_s$, and

$$\begin{eqnarray}
\tau_r \, \frac{dv(t)}{dt} = - v(t) + F\left[ I_s(t) \right]
\end{eqnarray}$$

with rate time-constant $\tau_r$ and activation function $F(s)$.

(Slide-24)

# How accurate are such models?

When output firing is approximately constant, a rate model with a single, fixed value of $\tau_\mathit{eff}$ provides a good approximation of a more detailed model.

<br>

When output firing varies greatly (and sometimes ceases altogether), the value of $\tau_\mathit{eff}$ must be adjusted (i.e., must change with $v$) in order to retain a good approximation.

<br>

In general, the value of $\tau_\mathit{eff}$ must be chosen appropriately for the case being studied.

(Slide-25)

# Adaptive rate model compared to detailed model

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_D&A-7-2.png" width="800">

D&A Fig-7-2

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_D&A-7-2-legend.png" width="400">

(Slide-26)

# Points to note

- We have related collective inputs and outputs of a single neuronal population.

- The description involves two successive low-pass filters

$$\begin{eqnarray}
\tau_s \, \frac{dI_s(t)}{dt} = -I_s(t) +  w_s \, u(t)
\\
\\
\tau_r \, \frac{dv(t)}{dt} = - v(t) + F\left[ I_s(t) \right]
\end{eqnarray}$$

- The first filter represents the synaptic time constant $\tau_s$, and the second filter represents the time-constant $\tau_r$ with which firing response follows membrane potential.

- The description also involves an activation function, defined in terms of the relation between a constant synaptic current and steady-state output firing.

$$\begin{eqnarray}
v_{ss}=F(I_s)
\end{eqnarray}$$

- Typically, this is a sigmoidal function, with threshold, inflection point, and saturation.

(Slide-27)

# **4. Firing rate models for interacting populations**

In order to model interacting populations, we seek an even more compact description that reduces our two dynamic equations

$$\begin{eqnarray}
\tau_s \, \frac{dI_s}{dt} = -I_s + w_s \, u,
\qquad\qquad
\tau_r \, \frac{dv}{dt} = -v + F(I_s)
\end{eqnarray}$$

to a single dynamic equation. A common trick for coupled dynamic equations is to distinguish between fast-changing and slow-changing state variables. Another common trick is to analyse **steady-states** where state variables have reached a stable value.

*Lecture 5 will introduce tools for analyzing dynamical systems.*

<br>

**Possibility A: Slow membrane, fast synapse**

When $\tau_r >> \tau_s$ (much larger), synaptic current $I_s$  may be replaced by its instantaneous steady-state value $w_s  u$:

$$\begin{eqnarray}
I_{s}(t)  \approx w_s u(t) \qquad\qquad\Rightarrow\qquad\qquad \tau_r \, \frac{dv}{dt} = -v + F\left( w_s \,  u \right)
\end{eqnarray}$$

which leaves us with one **non-linear** dynamic equation. This equation approximates the time-development of $v$ at time-scales *slower than* $\tau_s$.

The overall steady-state of the output rate is

$$\begin{eqnarray}
v_{ss}  = F\left(  w_s \, u \right)
\end{eqnarray}$$

This approximates time-scales slower than both $\tau_r$ and $\tau_s$.

**Possibility B: Slow synapse, fast membrane**

Alternatively, when $\tau_r<< \tau_s$ (much smaller), output rate $v$ may be replaced by its instantaneous equilibrium value $F(I_s)$, leaving

$$\begin{eqnarray}
\tau_s \, \frac{dI_s}{dt} = - I_s +  w\, u \qquad\qquad v \approx F( I_s )
\end{eqnarray}$$


Note that the remaining dynamic equation is *linear* in this approximation!  This has advantages for some theoretical analyses of network activity. This equation approximates the time-development of $v$ at time-scales *slower than* $\tau_r$.


As before, at time-scales slower than both $\tau_r$ and $\tau_s$, the steady-state value of output activity is

$$\begin{eqnarray}
v_{ss}  = F\left(  w_s \, u \right)
\end{eqnarray}$$

(Slide-28)


# Two alternative descriptions of firing populations



We can choose to describe the spiking activity of a neuronal population in terms of its output firing rate $v(t)$:

$$\begin{eqnarray}
\tau_r \, \frac{dv}{dt} = -v + F\left[ w_s \,  u(t) \right]\
\end{eqnarray}$$

Alternatively, we can choose to describe it in terms of its average synaptic current $I_s(t)$:

$$\begin{eqnarray}
\tau_s \, \frac{dI_s}{dt} = - I_s +  w_s\, u, \qquad\qquad v = F( I_s )
\end{eqnarray}$$

<br>

We are now ready to study the computational capabilities of networks of neuronal populations.

(Slide-29)

# Feedforward one-to-one

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_one2one_ff.png" height="200">

An input unit with activity $u$ drives an output unit with activity $v$.  The relation between input and output activities is approximated by a dynamic equation and the activation function $F()$:

$$\begin{eqnarray}
\tau \frac{d  v}{dt} = - v +  F \left(w \, u \right)
\end{eqnarray}$$

At steady-state, output activity is

$$\begin{eqnarray}
 v_{ss}  = F\left(  w \, u \right)
\end{eqnarray}$$

(Slide-30)

# Feedforward many-to-one

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_many2one_ff.png" height="200">

In a feedforward network, $N_u$ input populations $i$ with activities $u_i$ converge via synapses with weights $w_i$ on a single output population with activity $v$.   

For convenience, we may collect input activities and synaptic weights into vectors $\bf u$ and $\bf w$ and use the **vector dot product**.  The output rate is then given by


$$\begin{eqnarray}
\tau \frac{d v}{dt} = -v + F \left( \bf w \cdot \bf u \right) \qquad {\rm or} \qquad \tau_r \frac{dv}{dt} = - v + F \left( \sum_{i=1}^{N_u} w_{i} \, u_i \right)
\end{eqnarray}$$

At steady-state, the output firing rate is

$$\begin{eqnarray}
v_{ss}  = F\left(  \bf w \cdot \bf u \right)
\end{eqnarray}$$

(Slide-31)

# Feedforward many-to-many

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_many2many_ff.png" height="200">

In a feedforward network, $N_u$ input populations $i$ with activities $u_i$ converge via synapses with weights $w_{io}$ onto $N_v$ output populations $o$ with activites $v_o$.  For convenience, input and output activities are collected in vectors $\bf u$ and $\bf v$, and the synaptic weights into a matrix $\bf W$.  The output rate is then given by

$$\begin{eqnarray}
\tau \frac{d \bf v}{dt} = -{\bf v} + {\bf F} \left[ {\bf W} \cdot {\bf u} \right] \qquad {\rm or} \qquad \tau_r \frac{dv_o}{dt} = - v_o + F \left[ \sum_{i=1}^{N_u} w_{io} \, u_i \right]
\end{eqnarray}$$

At steady-state, the output activity is

$$\begin{eqnarray}
{\bf v}_\mathit{ss} = {\bf F}\left[  {\bf W} \cdot {\bf u} \right]
\end{eqnarray}$$

(Slide-32)


# Recurrent one-to-one

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_one2one_rec.png" height="200">

An input population with rate $u(t)$ drives an output population with rate $v(t)$, which also recurrently feeds back to itself.  The relation between input and output activities is described by

$$\begin{eqnarray}
\tau \frac{d  v}{dt} = - v +  F \left(w \, u + m \, v \right)
\end{eqnarray}$$

At steady-state, the output activity $v_{ss}$ has must satisfy

$$\begin{eqnarray}
 v_{ss}  = F\left(  w \, u + m \, v_{ss} \right)
\end{eqnarray}$$

Such non-linear equations can solved numerically. Sigmoidal activation functions admit up to **three steady-state values**, of which up to two values represent **stable steady-states**. We return to this topic in Lecture 5.

(Slide-33)

In [ ]:
from inspect import signature
# @title Steady-states of recurrent network {"vertical-output":true,"display-mode":"form"}
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from scipy.stats import poisson
from scipy.stats import norm
from matplotlib.colors import LinearSegmentedColormap

# histogram equalization for flat distribution
def ActivationFunction( x_i, nu, Vmax ):

    # generalised logistic function
    V_i = Vmax / (1 + np.exp( -x_i ) )**(1/nu)

    return V_i

def GetIntersections( v_i, vss_i ):

    # difference
    diff_i = v_i - vss_i;

    # slope
    slope_i = np.gradient( diff_i )

    # look for sign changes

    vss_k = np.zeros( 3 );
    slope_k = np.zeros( 3 );
    k = -1;

    for i in range( len(v_i)-1 ):

        if diff_i[i] * diff_i[i+1] < 0:

            k = k + 1;
            # linear interpolation
            alpha      = np.abs( diff_i[i] ) / np.abs( diff_i[i] - diff_i[i+1] );
            vss_k[k]   = v_i[i] + alpha * ( v_i[i+1] - v_i[i] );
            slope_k[k] = slope_i[i];


    return vss_k, slope_k

def plot_steady_state_activation( M=0.5, Wu=5, Vmax=10, Vthresh=9, nu=0.1 ):
    """(Slide-34)"""

    # range of output activities
    v_i = np.linspace( 0, Vmax, 1000);

    # argument of activation function
    x_i = M * ( v_i - Vthresh) + Wu;

    # output activity
    vss_i = ActivationFunction( x_i, nu, Vmax );

    # graphical solution of steady-state condition
    (vss_k, slope_k) = GetIntersections( v_i, vss_i )

    # Plot results
    fig, axs = plt.subplots( 1,1, figsize=(8,6))

    # first subplot
    axs.plot( v_i, vss_i, 'r-', linewidth=2 )
    axs.plot( v_i, v_i, 'k-', linewidth=2 )

    for k in range( len(vss_k)):
        if (vss_k[k] > 0) & (slope_k[k] > 0):
            axs.plot( vss_k[k], vss_k[k], 'o', color=(0.25, 0.75, 0.25), markersize=15, label='stable' )

    for k in range( len(vss_k)):
        if (vss_k[k] > 0) & (slope_k[k] < 0):
            axs.plot( vss_k[k], vss_k[k], 'ro', markersize=15, label='unstable' )

    axs.legend( fontsize=24);
    axs.set_xlim((0,Vmax))
    axs.set_xlabel('activity v')
    axs.set_box_aspect(1/1)
    axs.set_ylabel('v, F(v)')
    axs.set_title('Steady-state activity', fontsize=24);

    plt.tight_layout()
    plt.show()

# Creating sliders for concentrations and charge M, Wu, Vmax, Vthresh, nu
interact(plot_steady_state_activation, M = (0.1, 2.0, 0.1 ), Wu = (1, 10, 1), Vmax=(5, 20, 1), Vthresh=(5, 20, 1), nu=(0.01,0.2,0.01) )


interactive(children=(FloatSlider(value=0.5, description='M', max=2.0, min=0.1), IntSlider(value=5, descriptio…

<function __main__.plot_steady_state_activation(M=0.5, Wu=5, Vmax=10, Vthresh=9, nu=0.1)>

# Recurrent many-to-one

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_many2one_rec.png" height="200">

$N_u$ input populations with activities $u_i$ project with weights $w_i$ to a single output population with activity $v$.   The output population also projects to itself with weight $m$.

The time-development of output activity is described by

$$\begin{eqnarray}
\tau \frac{d v}{dt} = -v +  F \left( m \, v  + {\bf w} \cdot {\bf u} \right) \qquad \Leftrightarrow \qquad \tau_r \frac{dv}{dt} = - v + F \left( m \, v + \sum_{i=1}^{N_u} w_{i} \, u_i \right)
\end{eqnarray}$$

To compute the steady-state firing rate, $v_{ss}$, we need to solve

$$\begin{eqnarray}
 v_{ss}  = F\left(  m \, v_{ss} + {\bf w} \cdot {\bf u} \right)
\end{eqnarray}$$

Sigmoidal activation functions admit up to three steady-state values.

(Slide-35)

# Recurrent two-to-two

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_two2two_rec.png" height="200">

Two input populations with rate $u_{1,2}(t)$ drive two output populations with rate $v_{1,2}(t)$, which also recurrently feeds back to themselves and to each other.  The relation between input and output activities is described by

$$\begin{eqnarray}
\tau \,  \frac{d v_1}{dt}  &=& - v_1 + F\left[ w_1 u_1  + m_{11} v_1 + m_{12} v_2 \right]
\\
\tau \,  \frac{d v_2}{dt}  &=& - v_2 + F\left[ w_2 u_2  + m_{21} v_1 + m_{22} v_2 \right]
\end{eqnarray}$$

<br>

It is convenient to collect activities and synaptic weights into vectors and matrices

$$\begin{eqnarray}
\tau \, \left[ \begin{array}{c} \frac{d v_1}{dt} \\
\frac{d v_1}{dt}  \end{array}\right]
= -\left[ \begin{array}{c} v_1 \\ v_2  \end{array}\right]
+ F\left\{\left[\begin{array}{cc} w_1 & 0 \\ 0 & w_2 \end{array}\right] \cdot \left[\begin{array}{c} u_1 \\  u_2 \end{array}\right]
+\left[\begin{array}{cc} m_{11} & m_{12} \\  m_{21} & m_{22} \end{array}\right] \cdot \left[\begin{array}{c} v_1 \\ v_2 \end{array}\right]\right\}
\end{eqnarray}$$

<br>

and to take advantage of the compact notation of Linear Algebra:

$$\begin{eqnarray}
\tau \, \frac{d {\bf v}}{dt}= -{\bf v} + {\bf F}\left[{\bf W} \cdot {\bf u} + {\bf M} \cdot {\bf v} \right]
\end{eqnarray}$$

(Slide-36)

# Steady-state firing rate vector

Any steady-state firing rate vector must satisfy

$$\begin{eqnarray}
{\bf v}_{ss}  = {\bf F}\left(  {\bf W} \cdot {\bf u} + {\bf M} \cdot {\bf v}_{ss} \right)
\end{eqnarray}$$

<br>

In other words, for every output population $o$ with activity $v_o$, the excitatory and inhibitory projections from other populations must exactly balance the relaxation term $-v_o$, so that the time-derivative $\dot v_o$ is zero and activity $v_o$ remains stable.

<br>

Determining the existence and the values of such vectors requires more advanced methods, involving **eigenvectors and eigenvalues of recurrent connectivity** $\bf M$.   

<br>

We postpone this problem to Lectures 4 and 5.

(Slide-37)

# Recurrent many-to-many

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_many_rec.png" height="200">

$N_v$ input populations with rates $u_i$ project to $N_v$ output populatios with rates $v_o$, which also project to themselves and to each other with weights $w_{o'o}$.  The collective dynamic is described by:

<br>

$$\begin{eqnarray}
\tau \frac{d \bf v}{dt} = -{\bf v} + {\bf F} \left({\bf u} +  {\bf M} \cdot {\bf v} \right) \qquad
 \Leftrightarrow  \qquad \tau_r \frac{dv_o}{dt} = - v_o + F \left[ u_o +  \sum_{o'=1}^{N_v} m_{o'o} \, v_{o'} \right]
\end{eqnarray}$$

(Slide-37)

# Points to note


- In describing networks with multiple populations of neurons, **connectivity** plays a central role.  Typically, we collect all connection weights into a  **connectivity matrix**.

<br>

- In feedforward networks, all projections are in the same 'feedforward' direction:

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_many2many_ff.png" height="75">

<br>

- Recurrent networks involve both 'feedforward' and 'recurrent' connections.  Here we consider only **one-layer recurrent** networks, with recurrent connections only in the output layer:

<img src="https://raw.githubusercontent.com/jochen-braun/Theoretical-Neuroscience-I/main/SSLec01_many_rec.png" height="110">

<br>

- Feedforward and recurrent networks behave quite differently and require different mathematical methods for analysis.

(Slide-38)


# **Lecture 1 Overview:**

1. **Multi-unit and single-unit activity**

2. **Firing rate or 'population activity'**

2. **Firing rate model for one population**

3. **Firing rate models for interacting populations**


# **Outlook next lectures:**

- **Lecture 2:** Feedforward networks for classification and for coordinate transforms.

- **Lecture 3:** Recurrent networks for selective amplification.

- **Lecture 4:** Recurrent networks for associative memory.

- **Lecture 5:** State-space analysis of small recurrent networks.



# **Next:**

# **2. Feedforward Networks**


